# Lesson 4: Building a Multi-Document Agent

这个 Jupyter Notebook 文件演示了 如何构建一个能够跨多篇学术论文进行智能问答的多文档智能体（Multi-Document Agent）。它使用 LlamaIndex 框架实现，主要展示了两个核心功能：
1. 基础版：为3篇论文创建专用工具，构建能回答单篇论文问题的智能体
2. 增强版：为11篇论文创建工具库，引入工具检索机制实现跨论文的复杂问题解答

该智能体能自动识别用户问题涉及的论文，调用相应工具获取信息，最终生成准确回答。特别适合处理需要参考多篇技术文档的复杂查询。

## Setup

In [4]:
# ================== 第一部分：环境准备 ==================
from helper import get_openai_api_key, get_dashscope_api_key
import os
import nest_asyncio
# 应用异步支持（在Jupyter环境中必需）
nest_asyncio.apply()

## 1. Setup an agent over 3 papers

In [5]:
# ================== 第二部分：基础多文档智能体（3篇论文） ==================
# 定义3篇论文的下载链接和本地文件名
urls = [
    "https://openreview.net/pdf?id=VtmBAGCN7o",  # MetaGPT论文
    "https://openreview.net/pdf?id=6PmJoRfdaK",  # LongLoRA论文
    "https://openreview.net/pdf?id=hSyW5go0v8",  # Self-RAG论文
]

papers = [
    "metagpt.pdf",
    "longlora.pdf",
    "selfrag.pdf",
]

In [6]:
from llama_index.llms.openai_like import OpenAILike
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

# llm = OpenAI(model="gpt-3.5-turbo")
# 这里用了DashScope的大模型替代OpenAI模型，
# LlamaIndex支持多种LLM接口，DaskScope兼容OpenAI API，可以使用OpenAILike类调用
# LlamaIndex也有专门的DashScope支持包，具体见 https://developers.llamaindex.ai/python/examples/llm/dashscope/
llm = OpenAILike(
    api_key=get_dashscope_api_key(),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=True,
)   
Settings.llm = llm
# 如果没有传入 Embedding 模型，则创建一个默认的 OpenAI Embedding 模型。
# Embedding 模型用于将文本转化为数字向量，是 VectorStoreIndex 的核心。
# embed_model = embed_model or OpenAIEmbedding(model="text-embedding-ada-002")
model_real_path = os.path.expanduser(
    "~/Desktop/AIAgent/models/models--BAAI--bge-small-en-v1.5/snapshots/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a"
)
Settings.embed_model =  HuggingFaceEmbedding(
    model_name=model_real_path,
    # 默认情况下，LlamaIndex 会尝试自动下载和加载模型
    device="cpu"  # 如果您没有GPU，可以使用"cpu"
)

In [18]:
# 为每篇论文创建专用工具（向量检索工具 + 摘要工具）
import importlib  # 导入 Python 内置的 importlib 模块，该模块提供了对 import 机制的底层控制功能
import utils      # 导入你本地定义的工具模块（通常名为 utils.py），里面可能包含加载 API 密钥或初始化配置的函数
# 强制重新加载 utils 模块。
# 场景：如果你在 utils.py 里修改了一个函数，普通的 'import utils' 会因为缓存而不会生效。
# 使用 reload 可以让解释器重新读取该文件，确保你调用的是最新版本的代码。
importlib.reload(utils)
from utils import get_doc_tools
from pathlib import Path

paper_to_tools_dict = {}
for paper in papers:
    print(f"正在为论文创建工具: {paper}")
    # get_doc_tools函数会：
    # 1. 加载PDF文档
    # 2. 创建向量索引（用于语义搜索）
    # 3. 生成文档摘要
    # 返回两个工具：vector_tool（向量检索工具）和summary_tool（摘要工具）
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

正在为论文创建工具: metagpt.pdf
正在为论文创建工具: longlora.pdf
正在为论文创建工具: loftq.pdf
正在为论文创建工具: swebench.pdf
正在为论文创建工具: selfrag.pdf
正在为论文创建工具: zipformer.pdf
正在为论文创建工具: values.pdf
正在为论文创建工具: finetune_fair_diffusion.pdf
正在为论文创建工具: knowledge_card.pdf
正在为论文创建工具: metra.pdf
正在为论文创建工具: vr_mcl.pdf


UnicodeEncodeError: 'utf-8' codec can't encode character '\ud835' in position 956: surrogates not allowed

In [9]:
# 合并所有工具到一个列表
initial_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]

In [10]:
len(initial_tools)

6

In [11]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(
    tools=initial_tools,  # 传入所有工具
    llm=llm,        # 使用的语言模型
    verbose=True    # 开启详细日志（显示工具调用过程）
)

In [12]:
# 测试智能体（询问LongLoRA论文的评估数据集）
response = await agent.run(
    "Tell me about the evaluation dataset used in LongLoRA, "
    "and then tell me about the evaluation results"
)
print("\n-----------toll calls-----")
for tool_call in response.tool_calls:
    print(f"掉{tool_call.tool_name}返回:{tool_call.tool_output}")
print("\n-----------finnal response-----")
print(str(response))

2026-05-06 21:29:28,278 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the evaluation dataset used in LongL...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-06 21:29:28,278 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-06 21:29:28,281 - INFO - [init_run:0] complete with AgentInput
2026-05-06 21:29:28,282 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the evaluation dataset used in LongLoRA, and then tell ...
2026-05-06 21:29:28,282 - INFO - [setup_agent:0] started from AgentInput
2026-05-06 21:29:28,283 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-06 21:29:28,283 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the evaluation dataset used in LongLoRA, and then 

2026-05-06 21:29:29,692 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-06 21:29:31,791 - INFO - [run_agent_step:0] complete with AgentOutput
2026-05-06 21:29:31,792 - INFO - [tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [ChoiceDeltaToolCall(index=0, id='call_f62ba6022b754b049d2013', function=ChoiceDeltaTool...
2026-05-06 21:29:31,793 - INFO - [parse_agent_output:0] started from AgentOutput
2026-05-06 21:29:31,795 - INFO - [tick] add: ToolCall(tool_name='vector_tool_longlora', tool_kwargs={'query': 'evaluation dataset used in LongLoRA'}, tool_id='call_f62ba6022b754b049d2013')
2026-05-06 21:29:31,797 - INFO - [call_tool:0] started from ToolCall
2026-05-06 21:29:31,799 - INFO - [parse_agent_output:0] complete with no result
2026-05-06 21:29:31,804 - INFO - [tick] add: ToolCall(tool_name='vector_tool_longlora', tool_kwargs={'query': 'evaluatio


-----------toll calls-----
掉vector_tool_longlora返回:The evaluation datasets used in LongLoRA include the book corpus dataset PG19, the cleaned Arxiv Math proof-pile dataset, and long-context benchmarks such as LongBench and LEval. Additionally, there is a topic retrieval evaluation using a very long conversation dataset with varying context lengths (around 3k, 6k, 10k, 13k, and 16k).
掉vector_tool_longlora返回:The evaluation of LongLoRA shows promising results across different tasks and model sizes. For long-sequence language modeling, the perplexity improves as the context size increases. Specifically, for the Llama2 7B model, increasing the context window size from 8192 to 32768 tokens leads to a decrease in perplexity from 2.72 to 2.50. Similarly, for the Llama2 13B model, the perplexity reduction is even more significant, with a decrease of -0.28.

In terms of the maximum context length that can be fine-tuned on a single 8× A100 machine, LongLoRA extends the Llama2 7B, 13B, and 70B mo

2026-05-06 21:53:11,432 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-06 21:53:18,214 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-06 21:55:36,320 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-06 21:55:37,252 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-06 21:57:25,901 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-06 21:57:28,156 - INFO - HTTP Request: POST https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions "HTTP/1.1 200 OK"


In [13]:
# 测试跨论文查询
response = await agent.run("Give me a summary of both Self-RAG and LongLoRA")
print("\n-----------toll calls-----")
for tool_call in response.tool_calls:
    print(f"掉{tool_call.tool_name}返回:{tool_call.tool_output}")
print("\n-----------finnal response-----")
print(str(response))

2026-05-06 21:33:02,915 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Give me a summary of both Self-RAG and LongLoRA', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-06 21:33:02,915 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-06 21:33:02,918 - INFO - [init_run:0] complete with AgentInput
2026-05-06 21:33:02,918 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Give me a summary of both Self-RAG and LongLoRA')])], current_agent_n...
2026-05-06 21:33:02,919 - INFO - [setup_agent:0] started from AgentInput
2026-05-06 21:33:02,920 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-06 21:33:02,921 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Give me a summary of both Self-RAG and LongLoRA')])], current_agent_n.


-----------toll calls-----
掉summary_tool_selfrag返回:Error code: 400 - {'error': {'message': '<400> InternalError.Algo.InvalidParameter: Range of input length should be [1, 30720]', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_parameter_error'}, 'id': 'chatcmpl-498de031-4c49-960a-a229-9d24159ef7ef', 'request_id': '498de031-4c49-960a-a229-9d24159ef7ef'}
掉summary_tool_longlora返回:LongLoRA is an efficient fine-tuning approach designed to extend the context sizes of pre-trained large language models (LLMs) with minimal computational cost. It addresses the challenge of training LLMs with long context lengths, which is typically computationally expensive and resource-intensive. LongLoRA introduces two key innovations: Shifted Sparse Attention (S2-Attn) and an improved version of Low-Rank Adaptation (LoRA+). S2-Attn approximates standard self-attention by splitting the context into groups and shifting tokens, reducing computational costs while maintaining performance. LoRA+ 

## 2. Setup an agent over 11 papers (在11份文件上设置一个代理)

### Download 11 ICLR papers

In [19]:
# ================== 第三部分：增强版多文档智能体（11篇论文） ==================

urls = [
    "https://openreview.net/pdf?id=VtmBAGCN7o",  # MetaGPT
    "https://openreview.net/pdf?id=6PmJoRfdaK",  # LongLoRA
    "https://openreview.net/pdf?id=LzPWWPAdY4",  # LoftQ
    "https://openreview.net/pdf?id=VTF8yNQM66",  # SWE-Bench
    "https://openreview.net/pdf?id=hSyW5go0v8",  # Self-RAG
    "https://openreview.net/pdf?id=9WD9KwssyT",  # Zipformer
    "https://openreview.net/pdf?id=yV6fD7LYkF",  # Values
    "https://openreview.net/pdf?id=hnrB5YHoYu",  # Finetune Fair Diffusion
    "https://openreview.net/pdf?id=WbWtOYIzIK",  # Knowledge Card
    "https://openreview.net/pdf?id=c5pwL0Soay",  # Metra
    "https://openreview.net/pdf?id=TpD2aG1h0D"   # VR-MCL
]

papers = [
    "metagpt.pdf",
    "longlora.pdf",
    "loftq.pdf",
    "swebench.pdf",
    "selfrag.pdf",
    "zipformer.pdf",
    "values.pdf",
    "finetune_fair_diffusion.pdf",
    "knowledge_card.pdf",
    "metra.pdf",
    # "vr_mcl.pdf"
]

In [20]:
from utils import get_doc_tools
from pathlib import Path

# 为所有11篇论文创建工具（同上）
paper_to_tools_dict = {}
for paper in papers:
    print(f"Getting tools for paper: {paper}")
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

Getting tools for paper: metagpt.pdf
Getting tools for paper: longlora.pdf
Getting tools for paper: loftq.pdf
Getting tools for paper: swebench.pdf
Getting tools for paper: selfrag.pdf
Getting tools for paper: zipformer.pdf
Getting tools for paper: values.pdf
Getting tools for paper: finetune_fair_diffusion.pdf
Getting tools for paper: knowledge_card.pdf
Getting tools for paper: metra.pdf


### Extend the Agent with Tool Retrieval

In [21]:
all_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]
len(all_tools)

20

In [22]:
# ===== 关键改进：引入工具检索机制 =====
# 创建工具对象索引（将所有工具存储在向量数据库中）
# define an "object" index and retriever over these tools
from llama_index.core import VectorStoreIndex
from llama_index.core.objects import ObjectIndex

obj_index = ObjectIndex.from_objects(
    all_tools,                # 所有工具列表
    index_cls=VectorStoreIndex,  # 使用向量索引存储
)

In [24]:
# 创建工具检索器（可动态查找最相关的工具）
obj_retriever = obj_index.as_retriever(similarity_top_k=5)

In [25]:
# 测试工具检索（查询涉及MetaGPT和SWE-Bench的工具）
tools = obj_retriever.retrieve(
    "Tell me about the eval dataset used in MetaGPT and SWE-Bench"
)

In [35]:
print(f"检索到的工具数量: {len(tools)}")
print(f"工具元数据: {tools[3].metadata}")

检索到的工具数量: 5
工具元数据: ToolMetadata(description='Useful for summarization questions related to finetune_fair_diffusion', name='summary_tool_finetune_fair_diffusion', fn_schema=<class 'llama_index.core.tools.types.DefaultToolFnSchema'>, return_direct=False)


In [28]:

# 创建支持工具检索的智能体
agent = FunctionAgent(
    tool_retriever=obj_retriever,  # 使用工具检索器代替固定工具列表
    llm=llm,        # 使用的语言模型
    system_prompt=""" \
    You are an agent designed to answer queries over a set of given papers.
    Please always use the tools provided to answer a question. Do not rely on prior knowledge.\
    """,
    verbose=True    # 开启详细日志（显示工具调用过程）
)


In [29]:
tools = obj_retriever.retrieve(
    "Tell me about the evaluation dataset used "
    "in MetaGPT and compare it against SWE-Bench"
)
print(f"检索到的工具数量: {len(tools)}")
for tool in tools:
    print(f"工具元数据: {tool.metadata}")

检索到的工具数量: 5
工具元数据: ToolMetadata(description='vector_tool_metagpt(query: str, page_numbers: Optional[List[str]] = None) -> str\nUse to answer questions over a given paper.\n    \n        Useful if you have specific questions over the paper.\n        Always leave page_numbers as None UNLESS there is a specific page you want to search for.\n    \n        Args:\n            query (str): the string query to be embedded.\n            page_numbers (Optional[List[str]]): Filter by set of pages. Leave as NONE \n                if we want to perform a vector search\n                over all pages. Otherwise, filter by the set of specified pages.', name='vector_tool_metagpt', fn_schema=<class 'llama_index.core.tools.utils.vector_tool_metagpt'>, return_direct=False)
工具元数据: ToolMetadata(description='Useful for summarization questions related to metagpt', name='summary_tool_metagpt', fn_schema=<class 'llama_index.core.tools.types.DefaultToolFnSchema'>, return_direct=False)
工具元数据: ToolMetadata(descri

In [30]:
# 这个问题通义始终没有发起掉SWE-Bench相关检索工具的请求
response = await agent.run(
    "Tell me about the evaluation dataset used "
    "in MetaGPT and compare it against SWE-Bench"
)  #告诉我MetaGPT使用的评估数据集，并将其与SWE-Bench进行比较
print("\n-----------toll calls-----")
for tool_call in response.tool_calls:
    print(f"掉{tool_call.tool_name}：检索参数: {tool_call.tool_kwargs}")
    if hasattr(tool_call.tool_output.raw_output, "source_nodes"):
        for n in tool_call.tool_output.raw_output.source_nodes:
            print(f"元信息：{n.metadata}")
    print(f"返回:{tool_call.tool_output}")
print("\n-----------finnal response-----")
print(str(response))

2026-05-06 21:53:03,501 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the evaluation dataset used in MetaG...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-06 21:53:03,501 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-06 21:53:03,503 - INFO - [init_run:0] complete with AgentInput
2026-05-06 21:53:03,504 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the evaluation dataset used in MetaGPT and compare it a...
2026-05-06 21:53:03,504 - INFO - [setup_agent:0] started from AgentInput
2026-05-06 21:53:03,505 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-06 21:53:03,506 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='     You are an agent designed to answer queries over a set 


-----------toll calls-----
掉vector_tool_metagpt：检索参数: {'query': 'evaluation dataset used'}
元信息：{'page_label': '7', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '23', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '24', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '12', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'application/pdf', 'file_size': 16911937, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '8', 'file_name': 'metagpt.pdf', 'file_path': 'metagpt.pdf', 'file_type': 'app

In [31]:
# 改用openai的模型来对上个问题进行测试，发现openai模型能成功调用SWE-Bench相关工具
# 需要使用这个模型需要把utils.py里get_github_token方法里返回的token改成你自己的token
import importlib
import helper
importlib.reload(helper)
from helper import get_github_token
llm2 = OpenAILike(
    api_key=get_github_token(),
    api_base="https://models.inference.ai.azure.com/",
    model="gpt-4o-mini",
    temperature=0.1,
    context_window=128000,
    is_chat_model=True,
    is_function_calling_model=True,
)

agent2 = FunctionAgent(
    tool_retriever=obj_retriever,  # 使用工具检索器代替固定工具列表
    llm=llm,        # 使用的语言模型
    system_prompt=""" \
    You are an agent designed to answer queries over a set of given papers.
    Please always use the tools provided to answer a question. Do not rely on prior knowledge.\
    """,
    verbose=True    # 开启详细日志（显示工具调用过程）
)

response2 = await agent2.run(
    "Tell me about the evaluation dataset used "
    "in MetaGPT and compare it against SWE-Bench"
)  #告诉我MetaGPT使用的评估数据集，并将其与SWE-Bench进行比较
print("\n-----------toll calls-----")
for tool_call in response2.tool_calls:
    print(f"掉{tool_call.tool_name}：检索参数: {tool_call.tool_kwargs}")
    if hasattr(tool_call.tool_output.raw_output, "source_nodes"):
        for n in tool_call.tool_output.raw_output.source_nodes:
            print(f"元信息：{n.metadata}")
    print(f"返回:{tool_call.tool_output}")
print("\n-----------finnal response-----")
print(str(response2))

2026-05-06 21:55:27,400 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Tell me about the evaluation dataset used in MetaG...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-06 21:55:27,401 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-06 21:55:27,402 - INFO - [init_run:0] complete with AgentInput
2026-05-06 21:55:27,403 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Tell me about the evaluation dataset used in MetaGPT and compare it a...
2026-05-06 21:55:27,403 - INFO - [setup_agent:0] started from AgentInput
2026-05-06 21:55:27,404 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-06 21:55:27,405 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='     You are an agent designed to answer queries over a set 


-----------toll calls-----
掉vector_tool_swebench：检索参数: {'query': 'evaluation dataset used'}
元信息：{'page_label': '19', 'file_name': 'swebench.pdf', 'file_path': 'swebench.pdf', 'file_type': 'application/pdf', 'file_size': 2680380, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '23', 'file_name': 'swebench.pdf', 'file_path': 'swebench.pdf', 'file_type': 'application/pdf', 'file_size': 2680380, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '10', 'file_name': 'swebench.pdf', 'file_path': 'swebench.pdf', 'file_type': 'application/pdf', 'file_size': 2680380, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '4', 'file_name': 'swebench.pdf', 'file_path': 'swebench.pdf', 'file_type': 'application/pdf', 'file_size': 2680380, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '3', 'file_name': 'swebench.pdf', 'file_path': 'swebench.pdf', 'file_type

In [32]:
response = await agent.run(
    "Compare and contrast the LoRA papers (LongLoRA, LoftQ). "
    "Analyze the approach in each paper first. "
)  # 比较和对比LoRA论文（LongLoRA，LoftQ）。首先分析每篇论文中的方法。
print("\n-----------toll calls-----")
for tool_call in response.tool_calls:
    print(f"掉{tool_call.tool_name}：检索参数: {tool_call.tool_kwargs}")
    if hasattr(tool_call.tool_output.raw_output, "source_nodes"):
        for n in tool_call.tool_output.raw_output.source_nodes:
            print(f"元信息：{n.metadata}")
    print(f"返回:{tool_call.tool_output}")
print("\n-----------finnal response-----")
print(str(response))

2026-05-06 21:57:16,135 - INFO - [tick] add: AgentWorkflowStartEvent(user_msg='Compare and contrast the LoRA papers (LongLoRA, Lo...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
2026-05-06 21:57:16,135 - INFO - [init_run:0] started from AgentWorkflowStartEvent
2026-05-06 21:57:16,138 - INFO - [init_run:0] complete with AgentInput
2026-05-06 21:57:16,138 - INFO - [tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Compare and contrast the LoRA papers (LongLoRA, LoftQ). Analyze the a...
2026-05-06 21:57:16,139 - INFO - [setup_agent:0] started from AgentInput
2026-05-06 21:57:16,140 - INFO - [setup_agent:0] complete with AgentSetup
2026-05-06 21:57:16,141 - INFO - [tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='     You are an agent designed to answer queries over a set 


-----------toll calls-----
掉vector_tool_loftq：检索参数: {'query': 'What is the approach used in LoftQ?'}
元信息：{'page_label': '9', 'file_name': 'loftq.pdf', 'file_path': 'loftq.pdf', 'file_type': 'application/pdf', 'file_size': 366134, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '7', 'file_name': 'loftq.pdf', 'file_path': 'loftq.pdf', 'file_type': 'application/pdf', 'file_size': 366134, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '5', 'file_name': 'loftq.pdf', 'file_path': 'loftq.pdf', 'file_type': 'application/pdf', 'file_size': 366134, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '6', 'file_name': 'loftq.pdf', 'file_path': 'loftq.pdf', 'file_type': 'application/pdf', 'file_size': 366134, 'creation_date': '2025-10-20', 'last_modified_date': '2025-10-20'}
元信息：{'page_label': '15', 'file_name': 'loftq.pdf', 'file_path': 'loftq.pdf', 'file_type': 'application/pdf', 'file